# Bronze: POS Sales Ingestion
Ingests raw POS sales data into Bronze Lakehouse with audit columns and schema validation.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, input_file_name, col
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()

# Schema enforcement
schema = StructType([
    StructField('SaleID', StringType(), False),
    StructField('DateKey', IntegerType(), True),
    StructField('StoreID', StringType(), True),
    StructField('ProductID', StringType(), True),
    StructField('Quantity', IntegerType(), True),
    StructField('UnitPrice', DoubleType(), True),
    StructField('DiscountPct', DoubleType(), True),
    StructField('NetUnitPrice', DoubleType(), True),
    StructField('GrossSales', DoubleType(), True),
    StructField('NetSales', DoubleType(), True),
    StructField('COGS', DoubleType(), True),
    StructField('GrossMargin', DoubleType(), True),
    StructField('SaleChannel', StringType(), True),
    StructField('PaymentMethod', StringType(), True),
    StructField('Region', StringType(), True)
])

df_raw = (spark.read
    .option('header', True)
    .schema(schema)
    .csv('Files/pos_data/fact_sales.csv'))

df_bronze = (df_raw
    .withColumn('_ingested_at', current_timestamp())
    .withColumn('_source_file', input_file_name())
    .withColumn('_layer', lit('bronze'))
    .withColumn('_batch_id', lit(f'BATCH_{spark.sparkContext.applicationId[:8]}')))

row_count = df_bronze.count()
print(f'Ingesting {row_count:,} rows to Bronze Lakehouse')

(df_bronze.write
    .format('delta')
    .mode('append')
    .option('mergeSchema', True)
    .saveAsTable('bronze_lakehouse.fact_sales_raw'))

print(f'✅ {row_count:,} rows successfully written to bronze_lakehouse.fact_sales_raw')